# 🎨 AUTO-SCRIBE V2: TRUNG TÂM ĐIỀU KHIỂN & CẦU NỐI LOCAL TUNNEL
Hệ thống tự động hóa Whiteboard Animation với **Giao diện Web trực quan (Chạy 1 Cell)**:
- 🌐 **Hỗ Trợ Local Bridge (Cloudflare Tunnel)**: Kết nối với Laptop để phân tích kịch bản **KHÔNG CẦN GEMINI API KEY** (Miễn phí 100%, siêu nhẹ ~15MB RAM).
- 🎙️ **Whisper Voiceover**: Bóc tách chính xác từng câu thoại và thời lượng.
- 🤖 **AI Sinh Ảnh Doodle SVG (Pollinations + vtracer)**: Tự động vẽ ảnh đen trắng và biến thành SVG nếu Drive chưa có.
- 💾 **Tự Động Lưu Vào Drive `/image/f/gen/`**: Lưu trữ và đặt tên file chuẩn hóa theo từ khóa tiếng Anh.
- 📦 **Xuất File VideoScribe (.scribe) 1-Click**: Tải trực tiếp ngay trên giao diện Web.

## ⚡ BƯỚC 1: Cài Đặt Môi Trường & Kết Nối Google Drive (Chỉ chạy 1 lần)

In [ ]:
from google.colab import drive
import os
import sys

print("🔗 Đang yêu cầu quyền truy cập Google Drive...")
drive.mount('/content/drive')

print("⏳ Đang cài đặt thư viện lõi (Whisper, Gemini, vtracer, Gradio, Pillow, ffmpeg)...")
!apt-get install -y ffmpeg
!pip install -q openai-whisper google-genai requests vtracer Pillow gradio

print("✅ Đã cài đặt xong toàn bộ môi trường! Hãy chuyển sang BƯỚC 2 để mở Giao Diện.")

## 🎛️ BƯỚC 2: Khởi Chạy Giao Diện Web Điều Khiển Toàn Diện (All-In-One UI)
Chạy cell này để mở Giao diện Web tương tác. Bạn có thể dán link Local Bridge (hoặc Gemini API Key), chọn audio, xem trước ảnh, vẽ lại ảnh và xuất file VideoScribe ngay tại một chỗ!

In [ ]:
import os
import re
import json
import time
import random
import shutil
import zipfile
import subprocess
import urllib.request
import urllib.parse
from PIL import Image
import vtracer
import whisper
import requests
from google import genai
from google.genai import types
import gradio as gr

# --- CÁC HÀM XỬ LÝ LÕI ---
ASSETS_DIR = "assets"
os.makedirs(ASSETS_DIR, exist_ok=True)

def clean_slug(text):
    text = re.sub(r'[^a-zA-Z0-9\s_-]', '', text)
    text = re.sub(r'\s+', '_', text).strip('_').lower()
    return text[:40] if text else "doodle_icon"

def videoscribe_escape(s):
    s = s.replace('&', '&amp;')
    s = s.replace('<', '&lt;')
    s = s.replace('"', '&quot;')
    return s

def generate_doodle_svg(prompt_keyword, target_svg_path, drive_save_path=None):
    os.makedirs(os.path.dirname(os.path.abspath(target_svg_path)), exist_ok=True)
    ai_prompt = (
        f"clean black and white whiteboard doodle line art, minimalist sketch drawing of {prompt_keyword}, "
        f"continuous clean black ink outline on pure white background, minimal vector icon, no color, no shading, high contrast, clean strokes"
    )
    encoded_prompt = urllib.parse.quote(ai_prompt)
    seed = random.randint(1000, 999999)
    url = f"https://image.pollinations.ai/prompt/{encoded_prompt}?width=768&height=768&model=flux&nologo=true&seed={seed}"
    
    temp_img = target_svg_path + ".temp_dl"
    temp_png = target_svg_path + ".temp.png"
    success = False
    
    for attempt in range(3):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=30) as resp:
                with open(temp_img, 'wb') as f:
                    f.write(resp.read())
            
            with Image.open(temp_img) as img:
                img = img.convert('RGB')
                img.save(temp_png, 'PNG')
            
            vtracer.convert_image_to_svg_py(
                temp_png,
                target_svg_path,
                colormode='binary',
                hierarchical='stacked',
                mode='spline',
                filter_speckle=4,
                corner_threshold=60,
                length_threshold=4.0,
                max_iterations=10,
                splice_threshold=45,
                path_precision=3
            )
            success = True
            break
        except Exception as e:
            time.sleep(2)
        finally:
            if os.path.exists(temp_img): os.remove(temp_img)
            if os.path.exists(temp_png): os.remove(temp_png)
            
    if success and os.path.exists(target_svg_path) and os.path.getsize(target_svg_path) > 100:
        if drive_save_path:
            try:
                os.makedirs(os.path.dirname(os.path.abspath(drive_save_path)), exist_ok=True)
                shutil.copy(target_svg_path, drive_save_path)
            except Exception: pass
        return target_svg_path
    else:
        with open(target_svg_path, "w", encoding="utf-8") as f:
            f.write(f'<svg xmlns="http://www.w3.org/2000/svg" width="500" height="500"><rect width="500" height="500" fill="none" stroke="#000" stroke-width="4"/><text x="250" y="250" font-size="26" text-anchor="middle" fill="#000">{prompt_keyword}</text></svg>')
        return target_svg_path

def search_or_generate_svg(query, drive_search_dirs, drive_gen_dir, target_asset_path, used_files=None):
    if used_files is None: used_files = set()
    stop_words = {"vector", "illustration", "clipart", "transparent", "icon", "svg", "drawing", "the", "a", "an"}
    raw_words = re.sub(r'[^a-zA-Z0-9]', ' ', query).lower().split()
    query_words = set([w for w in raw_words if w not in stop_words and len(w) > 1])
    
    best_matches = []
    max_score = 0
    
    for s_dir in drive_search_dirs:
        if os.path.exists(s_dir):
            for root, dirs, files in os.walk(s_dir):
                for f in files:
                    if f.lower().endswith('.svg') or f.lower().endswith('.png'):
                        clean_n = re.sub(r'[^a-zA-Z0-9]', ' ', os.path.splitext(f)[0]).lower()
                        item_w = set(clean_n.split())
                        score = len(query_words.intersection(item_w))
                        if score > 0:
                            if " ".join(query_words) in clean_n: score += 2.0
                            full_p = os.path.join(root, f)
                            if score > max_score:
                                max_score = score
                                best_matches = [full_p]
                            elif score == max_score:
                                best_matches.append(full_p)
                                
    if best_matches and max_score >= 1.0:
        unused = [m for m in best_matches if m not in used_files]
        chosen = random.choice(unused) if unused else random.choice(best_matches)
        used_files.add(chosen)
        shutil.copy(chosen, target_asset_path)
        return target_asset_path, f"Drive: {os.path.basename(chosen)}"
    
    slug_n = clean_slug(query)
    drive_save = os.path.join(drive_gen_dir, f"{slug_n}.svg") if drive_gen_dir else None
    if drive_save and os.path.exists(drive_save):
        drive_save = os.path.join(drive_gen_dir, f"{slug_n}_{random.randint(100,999)}.svg")
        
    generate_doodle_svg(query, target_asset_path, drive_save)
    time.sleep(1.2)
    return target_asset_path, f"AI Mới: {os.path.basename(drive_save) if drive_save else 'Local'}"

def clean_json_response(text):
    text = text.strip()
    if text.startswith("```json"): text = text[7:]
    elif text.startswith("```"): text = text[3:]
    if text.endswith("```"): text = text[:-3]
    return text.strip()

def build_scribe_file(audio_path, metadata_path="scene_metadata.json"):
    if not os.path.exists(metadata_path): return None
    with open(metadata_path, "r", encoding="utf-8") as f:
        meta_data = json.load(f)

    BUILD_DIR = "build_scribe"
    if os.path.exists(BUILD_DIR): shutil.rmtree(BUILD_DIR)
    os.makedirs(BUILD_DIR, exist_ok=True)

    # Audio
    audio_dst = os.path.join(BUILD_DIR, "voiceover.mp3")
    if audio_path and os.path.exists(audio_path):
        try:
            subprocess.run(['ffmpeg', '-y', '-i', audio_path, '-ar', '44100', '-ac', '2', '-b:a', '192k', audio_dst], stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        except Exception:
            shutil.copy(audio_path, audio_dst)

    # Drawing XML
    drawing_xml = os.path.join(BUILD_DIR, "drawing.xml")
    with open(drawing_xml, "w", encoding="utf-8") as f:
        f.write('<?xml version="1.0" encoding="utf-8"?>\n')
        f.write('<drawing visualScale="1.0" canvasType="0" canvasColor="-1" bgFitMode="stretch" version="3.7.3103" resolution="1080" voiceoverVolume="100" soundtrackVolume="100">\n')
        
        xml_elements = []
        element_counter = 1000000000 + random.randint(10000, 99999)
        visual_timeline_ms = 0
        
        for scene_idx, scene in enumerate(meta_data):
            raw_images = scene.get('images', [])
            n = len(raw_images)
            if n == 0: continue
            
            speech_dur_s = scene['end'] - scene['start']
            duration_per_img = speech_dur_s / n
            scene_x = scene_idx * 1600
            scene_y = 0
            cam_scale = 0.82
            cam_x = 448.5 - scene_x * cam_scale
            cam_y = 252.5 - scene_y * cam_scale
            
            for i, img_meta in enumerate(raw_images):
                filename = img_meta.get('file_name', '')
                file_path = os.path.join(ASSETS_DIR, filename)
                alt_svg = os.path.splitext(file_path)[0] + ".svg"
                if os.path.exists(alt_svg): file_path = alt_svg
                elif not os.path.exists(file_path): continue
                
                is_svg = file_path.endswith('.svg')
                actual_filename = os.path.basename(file_path)
                
                if not is_svg:
                    vec_svg = file_path + ".vectorized.svg"
                    try:
                        vtracer.convert_image_to_svg_py(file_path, vec_svg, colormode='binary', hierarchical='stacked', mode='spline')
                        file_path = vec_svg
                    except Exception: pass

                with open(file_path, "r", encoding="utf-8") as f2:
                    raw_svg = f2.read()
                    raw_svg = re.sub(r'<\?xml[^>]*\?>', '', raw_svg)
                    raw_svg = re.sub(r'<!DOCTYPE[^>]*>', '', raw_svg)
                    raw_svg = re.sub(r'<!--.*?-->', '', raw_svg, flags=re.DOTALL)
                    content = raw_svg.replace('\n', ' ').replace('\r', '')
                    
                element_counter += random.randint(1000, 5000)
                if n == 1: pos_x, pos_y, scale_val = scene_x, scene_y, "0.8"
                elif n == 2: pos_x, pos_y, scale_val = scene_x + (-250 if i == 0 else 250), scene_y, "0.55"
                else: pos_x, pos_y, scale_val = scene_x + (-220 if i == 1 else (220 if i == 2 else 0)), scene_y + (120 if i > 0 else -120), "0.45"
                    
                ai_style = img_meta.get('animation_style', 'draw')
                if ai_style == 'draw': draw_style = 'draw_style_normal'
                elif ai_style in ['movein', 'movein_hand', 'movein_nohand']: draw_style = 'draw_style_movein'
                elif ai_style == 'fadein': draw_style = 'draw_style_fadein'
                else: draw_style = 'draw_style_normal'
                
                movin_compass = str(random.randint(1, 8))
                draw_detail = 'yes' if draw_style == 'draw_style_normal' else 'no'
                custom_hand = 'default_nohand' if draw_style == 'draw_style_movein' else ''
                movin_arc = random.choice(['0', '1']) if draw_style == 'draw_style_movein' else '0'
                
                total_time_ms = int(duration_per_img * 1000)
                trans_time_ms = min(500, int(total_time_ms * 0.15))
                pause_time_ms = min(500, int(total_time_ms * 0.10))
                target_time_ms = max(0, total_time_ms - trans_time_ms - pause_time_ms)
                
                drawing_xml_attr = f'drawingXML="{videoscribe_escape(content)}"' if is_svg else f'drawingXML="{videoscribe_escape(content)}" imageRef="{actual_filename}"'
                
                element_xml = (
                    f'  <element elementType="drawing" descName="" elementID="{element_counter}" '
                    f'splitTextField="no" drawingText="" fontName="null" {drawing_xml_attr} '
                    f'customHandMD5="{custom_hand}" colourEffect="0" targetTime="{target_time_ms}" '
                    f'pauseTime="{pause_time_ms}" transitionTime="{trans_time_ms}" '
                    f'drawStyle="{draw_style}" rotation="0" visible="true" '
                    f'currentPosX="{pos_x}" currentPosY="{pos_y}" offsetX="{pos_x}" offsetY="{pos_y}" '
                    f'scalesX="0.8" scalesY="0.8" theScale="0.8" targetHeight="800" '
                    f'movinCompass="{movin_compass}" movinFlow="0" movinArc="{movin_arc}" movinAllowRotate="yes" '
                    f'drawDetail="{draw_detail}" sketchStyle="no" brush="0" brushOptions="0" opacity="1" '
                    f'textColour="-1" textAlign="left" textBackwards="no" rtlLanguage="no" textSpacing="0" '
                    f'flipHoriz="no" flipVert="no" locked="no" calligraphy_angle="45" keepRunning="no" '
                    f'loopOptions="Fit to Time" blendMode="normal" filters="&lt;filters/>" morphFromID="0" '
                    f'morphCamera="no" morphRemoveOld="yes" cameraPositionX="{cam_x}" cameraPositionY="{cam_y}" '
                    f'cameraScale="{cam_scale}" cameraCanvasWid="897.7777777777778" cameraCanvasHei="505" '
                    f'availableRecolours="&lt;availableRecolours/>" recolouringSchemes="&lt;recolouringSchemes/>" '
                    f'skinTone="-1" hairColour="-1" highlightColour="-1" customColour1="-1" customColour2="-1" '
                    f'originalOutlineColour="0" greyscaleContrast="70" />\n'
                )
                xml_elements.append(element_xml)
                visual_timeline_ms += total_time_ms
                
        f.write('\n'.join(xml_elements) + '\n')
        f.write('</drawing>')

    out_file = "Auto_Project.scribe"
    with zipfile.ZipFile(out_file, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(BUILD_DIR):
            for file in files:
                f_p = os.path.join(root, file)
                zipf.write(f_p, os.path.relpath(f_p, BUILD_DIR))

    # Fix > entity
    tmp_p = out_file + ".tmp"
    with zipfile.ZipFile(out_file, 'r') as zin, zipfile.ZipFile(tmp_p, 'w', zipfile.ZIP_DEFLATED) as zout:
        for item in zin.infolist():
            data = zin.read(item.filename)
            if item.filename == 'drawing.xml':
                data = data.decode('utf-8').replace('&gt;', '>').encode('utf-8')
            zout.writestr(item, data)
    os.replace(tmp_p, out_file)
    return out_file

# --- HÀM THỰC THI TOÀN BỘ 1-CLICK ---
def run_full_pipeline(audio_file_obj, bridge_url_input, api_key_input, drive_f_path, drive_gen_path, sec_per_img, progress=gr.Progress()):
    logs = []
    def log(msg):
        logs.append(msg)
        return "\n".join(logs)

    # 1. Kiểm tra audio
    progress(0.05, desc="Đang kiểm tra âm thanh...")
    audio_path = None
    if audio_file_obj is not None:
        audio_path = audio_file_obj if isinstance(audio_file_obj, str) else audio_file_obj.name
    elif os.path.exists("voiceover.mp3"):
        audio_path = "voiceover.mp3"
        
    if not audio_path or not os.path.exists(audio_path):
        return log("❌ Lỗi: Vui lòng upload hoặc chọn file voiceover.mp3!"), None, None

    log(f"🎙️ Âm thanh: {os.path.basename(audio_path)}")

    # 2. Whisper
    progress(0.15, desc="Đang chạy Whisper bóc tách thời gian...")
    log("⏳ Whisper đang nhận diện giọng đọc...")
    whisper_model = whisper.load_model("base")
    whisper_res = whisper_model.transcribe(audio_path)
    scenes = [{"start": seg["start"], "end": seg["end"], "text": seg["text"].strip()} for seg in whisper_res["segments"] if seg["text"].strip()]
    log(f"✅ Bóc tách thành công {len(scenes)} câu thoại!")

    # 3. Phân tích kịch bản: Ưu tiên Local Bridge URL (0 API Key) -> Fallback Gemini API
    progress(0.35, desc="AI đang phân tích kịch bản...")
    raw_analyzed_data = None
    bridge_url = bridge_url_input.strip().rstrip('/') if bridge_url_input else ""
    
    if bridge_url:
        log(f"🌐 Đang kết nối tới Local Bridge: {bridge_url} (0 API Key)...")
        try:
            resp = requests.post(f"{bridge_url}/analyze_scenes", json={"scenes": scenes, "sec_per_img": float(sec_per_img)}, timeout=30)
            if resp.status_code == 200:
                res_json = resp.json()
                raw_analyzed_data = res_json.get("data", [])
                log(f"🎉 Nhận kịch bản thành công từ Local Bridge ({len(raw_analyzed_data)} cảnh)!")
            else:
                log(f"⚠️ Local Bridge trả về lỗi {resp.status_code}")
        except Exception as e:
            log(f"⚠️ Không thể kết nối tới Local Bridge ({e}).")
            
    if not raw_analyzed_data and api_key_input.strip():
        log("🔑 Đang phân tích kịch bản bằng Gemini API...")
        client = genai.Client(api_key=api_key_input.strip())
        raw_analyzed_data = []
        BATCH_SIZE = 4
        for i in range(0, len(scenes), BATCH_SIZE):
            batch_scenes = scenes[i:i+BATCH_SIZE]
            prompt_text = ""
            for j, s in enumerate(batch_scenes):
                dur = s['end'] - s['start']
                num_imgs = max(1, int(round(dur / float(sec_per_img))))
                prompt_text += f"ID: {i+j+1} | CẦN {num_imgs} ẢNH | Thời lượng: {dur:.1f}s | Câu: \"{s['text']}\"\n"
                
            prompt_instr = f"""
            Bạn là Đạo diễn Whiteboard Animation. Hãy phân tích các câu thoại sau và tạo danh sách hình ảnh doodle phù hợp:
            {prompt_text}
            Trả về DUY NHẤT 1 MẢNG JSON ARRAY:
            [ {{"sentence_id": ID câu, "images": [ {{"visual_concept": "Ý tưởng", "svg_search_prompt": "Từ khóa tiếng Anh 1-3 từ", "animation_style": "draw", "movein", hoặc "fadein"}} ]}} ]
            """
            try:
                resp = client.models.generate_content(model="gemini-3.1-flash-lite", contents=prompt_instr, config=types.GenerateContentConfig(response_mime_type="application/json"))
                b_data = json.loads(clean_json_response(resp.text))
                raw_analyzed_data.extend(b_data)
            except Exception as e:
                log(f"⚠️ Lỗi Gemini batch {i//BATCH_SIZE+1}: {e}")

    # Fallback mặc định nếu cả 2 đều trống
    if not raw_analyzed_data:
        log("💡 Sử dụng bộ phân tích từ khóa mặc định tích hợp sẵn...")
        raw_analyzed_data = []
        for s_idx, s in enumerate(scenes):
            dur = s['end'] - s['start']
            num_imgs = max(1, int(round(dur / float(sec_per_img))))
            imgs = [{"visual_concept": "Ý tưởng", "svg_search_prompt": "concept idea", "animation_style": "draw"} for _ in range(num_imgs)]
            raw_analyzed_data.append({"sentence_id": s_idx + 1, "images": imgs})

    # 4. Sinh / Rút ảnh SVG Doodle & Đồng bộ Drive f/gen
    progress(0.55, desc="Đang sinh ảnh SVG Doodle & Lưu Drive f/gen...")
    log("🎨 Đang chuẩn bị toàn bộ ảnh SVG Doodle & Đồng bộ vào Drive f/gen/...")
    
    scene_metadata = []
    used_files = set()
    drive_search_dirs = [drive_f_path, drive_gen_path]
    os.makedirs(drive_gen_path, exist_ok=True)

    for j, s in enumerate(scenes):
        sc_id = j + 1
        res_item = next((it for it in raw_analyzed_data if it.get("sentence_id") == sc_id), None)
        raw_imgs = res_item.get("images", []) if res_item else []
        if not raw_imgs:
            raw_imgs = [{"visual_concept": "Minh họa", "svg_search_prompt": "concept idea", "animation_style": "draw"}]
            
        sentence_entry = {"sentence_id": sc_id, "start": s['start'], "end": s['end'], "speech_text": s['text'], "images": []}
        for img_idx, img_info in enumerate(raw_imgs):
            file_base = f"sentence_{sc_id:03d}_img_{img_idx+1:02d}"
            kw = img_info.get("svg_search_prompt", "icon")
            target_p = os.path.join(ASSETS_DIR, f"{file_base}.svg")
            
            act_p, src_note = search_or_generate_svg(kw, drive_search_dirs, drive_gen_path, target_p, used_files)
            
            sentence_entry["images"].append({
                "img_idx": img_idx + 1,
                "visual_concept": img_info.get("visual_concept", ""),
                "svg_search_prompt": kw,
                "animation_style": img_info.get("animation_style", "draw"),
                "file_name": os.path.basename(act_p),
                "source": src_note
            })
        scene_metadata.append(sentence_entry)
        
    with open("scene_metadata.json", "w", encoding="utf-8") as f:
        json.dump(scene_metadata, f, ensure_ascii=False, indent=2)
    log("🎉 Đã hoàn tất chuẩn bị toàn bộ ảnh SVG!")

    # 5. Xuất file VideoScribe
    progress(0.85, desc="Đang đóng gói file VideoScribe...")
    log("📦 Đang đóng gói dự án Auto_Project.scribe...")
    out_scribe = build_scribe_file(audio_path, "scene_metadata.json")
    
    progress(1.0, desc="Hoàn tất!")
    log("✨ HOÀN TẤT 100%! Bạn có thể tải file .scribe ở khung bên phải.")
    
    preview_html = generate_preview_table(scene_metadata)
    return "\n".join(logs), out_scribe, preview_html

def generate_preview_table(meta_data=None):
    if meta_data is None:
        if not os.path.exists("scene_metadata.json"): return "<p>Chưa có dữ liệu kịch bản.</p>"
        with open("scene_metadata.json", "r", encoding="utf-8") as f:
            meta_data = json.load(f)
            
    rows = ""
    for s in meta_data:
        imgs_div = ""
        for img in s.get('images', []):
            fp = os.path.join(ASSETS_DIR, img.get('file_name', ''))
            svg_content = ""
            if os.path.exists(fp) and fp.endswith('.svg'):
                with open(fp, "r", encoding="utf-8") as svg_f:
                    svg_content = svg_f.read()
                    svg_content = re.sub(r'<\?xml[^>]*\?>', '', svg_content)
                    
            imgs_div += f'''
            <div style="background:#fff; border:1px solid #e2e8f0; border-radius:6px; padding:6px; margin:4px; display:inline-block; vertical-align:top; width:110px; text-align:center;">
                <div style="height:70px; display:flex; align-items:center; justify-content:center; overflow:hidden;">{svg_content if svg_content else '<div style="color:#a0aec0;">IMG</div>'}</div>
                <div style="font-size:11px; font-weight:bold; color:#2d3748; overflow:hidden; text-overflow:ellipsis; white-space:nowrap;">{img.get('svg_search_prompt','')}</div>
                <div style="font-size:10px; color:#4a5568;">{img.get('animation_style','draw')}</div>
                <div style="font-size:9px; color:#3182ce;">{img.get('source','')}</div>
            </div>
            '''
        rows += f'''
        <tr style="border-bottom:1px solid #edf2f7;">
            <td style="padding:10px; font-weight:bold; color:#4a5568; vertical-align:top; width:50px;">#{s['sentence_id']}</td>
            <td style="padding:10px; vertical-align:top; width:80px; font-size:12px; color:#718096;">{s['start']:.1f}s ➔ {s['end']:.1f}s</td>
            <td style="padding:10px; vertical-align:top; color:#2d3748; font-size:13px; line-height:1.4;">{s['speech_text']}</td>
            <td style="padding:10px; vertical-align:top;">{imgs_div}</td>
        </tr>
        '''
    return f'''
    <div style="font-family:-apple-system, sans-serif; background:#fff; border-radius:8px; border:1px solid #e2e8f0; max-height:450px; overflow-y:auto;">
        <table style="width:100%; border-collapse:collapse; text-align:left;">
            <thead>
                <tr style="background:#f7fafc; color:#718096; font-size:11px; text-transform:uppercase; border-bottom:2px solid #e2e8f0;">
                    <th style="padding:10px;">ID</th><th style="padding:10px;">Thời Gian</th><th style="padding:10px;">Câu Thoại</th><th style="padding:10px;">Ảnh SVG & Hiệu Ứng</th>
                </tr>
            </thead>
            <tbody>{rows}</tbody>
        </table>
    </div>
    '''

def redraw_single_scene(scene_id, custom_kw, drive_gen_path):
    if not os.path.exists("scene_metadata.json"): return "Chưa có dữ liệu!", None
    with open("scene_metadata.json", "r", encoding="utf-8") as f:
        meta = json.load(f)
        
    found = False
    for s in meta:
        if s['sentence_id'] == int(scene_id):
            found = True
            for img_idx, img in enumerate(s.get('images', [])):
                kw = custom_kw.strip() if custom_kw.strip() else img.get('svg_search_prompt', 'concept')
                target_p = os.path.join(ASSETS_DIR, f"sentence_{int(scene_id):03d}_img_{img_idx+1:02d}.svg")
                slug_n = clean_slug(kw)
                drive_p = os.path.join(drive_gen_path, f"{slug_n}_{random.randint(100,999)}.svg")
                generate_doodle_svg(kw, target_p, drive_p)
                img['svg_search_prompt'] = kw
                img['source'] = f"AI Vẽ Lại: {os.path.basename(drive_p)}"
                
    if found:
        with open("scene_metadata.json", "w", encoding="utf-8") as f:
            json.dump(meta, f, ensure_ascii=False, indent=2)
        build_scribe_file("voiceover.mp3", "scene_metadata.json")
        return f"✅ Đã vẽ lại thành công cảnh #{scene_id} với từ khóa '{custom_kw}'!", generate_preview_table(meta)
    return f"❌ Không tìm thấy cảnh #{scene_id}", None

# --- XÂY DỰNG GIAO DIỆN WEB GRADIO (THEME HIỆN ĐẠI) ---
with gr.Blocks(title="Auto-Scribe V2 Control Center", theme=gr.themes.Soft(primary_hue="blue", neutral_hue="slate")) as app:
    gr.Markdown("# 🚀 AUTO-SCRIBE V2: TRUNG TÂM ĐIỀU KHIỂN & XUẤT VIDEO TRỰC QUAN")
    gr.Markdown("Hệ thống tự động hóa làm video Whiteboard: Tách giọng nói ➔ Lên kịch bản (0 API Key) ➔ Tự sinh SVG Doodle & Lưu Drive `f/gen/` ➔ Đóng gói VideoScribe.")
    
    with gr.Tabs():
        with gr.TabItem("🎛️ 1-Click Pipeline (Tự Động Toàn Bộ)"):
            with gr.Row():
                with gr.Column(scale=1):
                    audio_in = gr.File(label="🎙️ File Giọng Đọc (voiceover.mp3)", file_types=[".mp3", ".wav"])
                    bridge_url_in = gr.Textbox(label="🔗 Local Bridge URL (Cloudflare Tunnel) - KHUYÊN DÙNG (0 API KEY)", placeholder="https://xxxx.trycloudflare.com")
                    api_key_in = gr.Textbox(label="🔑 Hoặc Gemini API Key (Tùy chọn nếu không dùng Bridge)", type="password", placeholder="AIzaSy...")
                    drive_f_in = gr.Textbox(label="📂 Thư mục ảnh gốc trên Drive", value="/content/drive/MyDrive/image/f")
                    drive_gen_in = gr.Textbox(label="💾 Thư mục lưu ảnh AI sinh mới trên Drive", value="/content/drive/MyDrive/image/f/gen")
                    speed_slider = gr.Slider(label="⏱️ Thời lượng mỗi ảnh (giây)", minimum=2.0, maximum=6.0, value=3.5, step=0.5)
                    btn_run = gr.Button("🚀 BẮT ĐẦU TỰ ĐỘNG TOÀN BỘ (1-CLICK RUN)", variant="primary", size="lg")
                
                with gr.Column(scale=1):
                    log_box = gr.Textbox(label="📋 Nhật Ký Hoạt Động (Live Console Logs)", lines=12, interactive=False)
                    out_file = gr.File(label="📦 Tải File Dự Án VideoScribe (.scribe)")
                    
        with gr.TabItem("🖼️ Xem Trước & Sửa Ảnh Trực Quan (Visual Editor)"):
            preview_display = gr.HTML(label="Bảng Kịch Bản")
            with gr.Row():
                scene_select = gr.Number(label="ID Cảnh Muốn Vẽ Lại (Ví dụ: 1)", value=1, precision=0)
                custom_kw_in = gr.Textbox(label="Từ Khóa Vẽ Lại Mới (Tiếng Anh)", placeholder="businessman handshake, luxury watch...")
                btn_redraw = gr.Button("🎨 AI Vẽ Lại Cảnh Này & Lưu Drive", variant="secondary")
            redraw_msg = gr.Textbox(label="Thông Báo", interactive=False)
            
    # Sự kiện
    btn_run.click(
        fn=run_full_pipeline,
        inputs=[audio_in, bridge_url_in, api_key_in, drive_f_in, drive_gen_in, speed_slider],
        outputs=[log_box, out_file, preview_display]
    )
    
    btn_redraw.click(
        fn=redraw_single_scene,
        inputs=[scene_select, custom_kw_in, drive_gen_in],
        outputs=[redraw_msg, preview_display]
    )

print("🌐 Đang khởi chạy Giao Diện Web & Mở Đường Link Tunnel...")
app.queue().launch(share=True, debug=False, show_error=True)